# 05-01 指数加权移动平均

指数加权移动平均，常见英文是 Exponentially Weighted Moving Average，也常简称 EMA。

它不是神经网络独有的概念，但在深度学习里非常常见。Momentum、RMSProp、Adam、BatchNorm 的 running mean / running variance，都和它有关。

参考黑马程序员深度学习课程的学习主线，这个概念应该放在优化器之前学，因为 Momentum 和 Adam 里的“动量”“一阶矩”“二阶矩”本质上都离不开指数加权移动平均。

## 1. 为什么需要移动平均

训练神经网络时，很多量都会一会儿大、一会儿小。

比如每个 batch 算出来的梯度：

$$
g_1,g_2,g_3,\dots,g_t
$$

因为每个 batch 的样本不同，所以梯度可能很抖。

如果我们只看当前梯度 $g_t$，容易被当前 batch 的偶然性影响。

移动平均想解决的问题是：**不要只看当前值，也参考最近一段时间的整体趋势。**

## 2. 普通平均有什么问题

如果有 $t$ 个数：

$$
x_1,x_2,\dots,x_t
$$

普通平均是：

$$
\bar{x}_t=\frac{1}{t}\sum_{i=1}^{t}x_i
$$

它的问题是：很早之前的数据和当前数据权重一样。

但训练神经网络时，我们通常更关心最近的趋势。

比如第 $1$ 步的梯度，对第 $10000$ 步的参数更新来说，参考价值可能已经很小。

所以我们希望：

```text
最近的数据权重大
很久以前的数据权重小
```

## 3. 指数加权移动平均的公式

指数加权移动平均的核心公式是：

$$
v_t=\beta v_{t-1}+(1-\beta)x_t
$$

其中：

- $x_t$ 是当前时刻的新值。
- $v_t$ 是当前时刻的移动平均结果。
- $v_{t-1}$ 是上一时刻的移动平均结果。
- $\beta$ 是衰减系数，通常在 $0$ 到 $1$ 之间。

这句话可以读成：

```text
新的平均值 = 一部分旧趋势 + 一部分当前值
```

比如 $\beta=0.9$ 时：

$$
v_t=0.9v_{t-1}+0.1x_t
$$

这表示当前结果里，$90\%$ 来自过去趋势，$10\%$ 来自当前新值。

## 4. 为什么叫“指数加权”

这个名字不是随便来的。把公式展开，就能看出来过去数据的权重是按指数衰减的。

从：

$$
v_t=\beta v_{t-1}+(1-\beta)x_t
$$

继续展开 $v_{t-1}$：

$$
v_{t-1}=\beta v_{t-2}+(1-\beta)x_{t-1}
$$

代回去：

$$
v_t=\beta[\beta v_{t-2}+(1-\beta)x_{t-1}]+(1-\beta)x_t
$$

整理：

$$
v_t=\beta^2v_{t-2}+\beta(1-\beta)x_{t-1}+(1-\beta)x_t
$$

继续展开，可以得到近似形式：

$$
v_t\approx(1-\beta)x_t+(1-\beta)\beta x_{t-1}+(1-\beta)\beta^2x_{t-2}+\cdots
$$

可以看到：

- 当前值 $x_t$ 的权重是 $(1-\beta)$。
- 上一个值 $x_{t-1}$ 的权重是 $(1-\beta)\beta$。
- 再上一个值 $x_{t-2}$ 的权重是 $(1-\beta)\beta^2$。

越久远的数据，乘上的 $\beta$ 次数越多，权重越小。

这就是“指数加权”的来源。

## 5. $\beta$ 到底控制什么

$\beta$ 控制记忆长度。

如果 $\beta$ 很小，比如：

$$
\beta=0.5
$$

那么当前值占比是：

$$
1-\beta=0.5
$$

这说明移动平均很在意当前值，变化会比较快，但也比较抖。

如果 $\beta$ 很大，比如：

$$
\beta=0.99
$$

当前值占比只有：

$$
1-\beta=0.01
$$

这说明移动平均非常相信过去趋势，曲线更平滑，但对新变化反应更慢。

所以：

| $\beta$ | 记忆效果 | 曲线特点 |
|---|---|---|
| 小 | 记得短 | 反应快，但更抖 |
| 大 | 记得长 | 更平滑，但反应慢 |

常见经验是：

$$
\beta=0.9
$$

大约可以理解成参考最近 $10$ 个左右的数据趋势，因为：

$$
\frac{1}{1-\beta}=\frac{1}{1-0.9}=10
$$

如果 $\beta=0.99$，则大约参考最近 $100$ 个左右的数据趋势。

## 6. 为什么一开始会有偏差

指数加权移动平均通常从：

$$
v_0=0
$$

开始。

这会带来一个问题：刚开始的 $v_t$ 会偏小。

例如：

$$
v_1=\beta v_0+(1-\beta)x_1
$$

因为 $v_0=0$，所以：

$$
v_1=(1-\beta)x_1
$$

如果 $\beta=0.9$：

$$
v_1=0.1x_1
$$

明明第一个观测值是 $x_1$，但移动平均只有 $0.1x_1$，明显偏小。

这是因为一开始没有历史数据，但公式默认历史趋势 $v_0=0$，把结果往 $0$ 拉了。

## 7. 偏差修正是什么

为了解决前期偏小的问题，可以做偏差修正：

$$
\hat{v}_t=\frac{v_t}{1-\beta^t}
$$

这里 $t$ 是当前步数。

为什么这样能修正？

因为刚开始时，真正分配给已有数据的权重总和还没有接近 $1$。$1-\beta^t$ 可以理解成当前已经积累起来的有效权重总量。

例如 $t=1$ 时：

$$
1-\beta^1=1-\beta
$$

而：

$$
v_1=(1-\beta)x_1
$$

修正后：

$$
\hat{v}_1=\frac{(1-\beta)x_1}{1-\beta}=x_1
$$

这就把刚开始被压小的问题修回来了。

Adam 里的一阶矩和二阶矩都会使用类似的偏差修正。

## 8. EMA 和 Momentum 的关系

Momentum 本质上就是对梯度做指数加权移动平均。

普通 SGD 直接用当前梯度：

$$
\theta_t=\theta_{t-1}-\eta g_t
$$

Momentum 先维护一个梯度趋势：

$$
v_t=\beta v_{t-1}+(1-\beta)g_t
$$

再更新参数：

$$
\theta_t=\theta_{t-1}-\eta v_t
$$

这样参数更新不只看当前 batch 的梯度，而是看一段时间的平均方向。

这就是 Momentum 能减少震荡的原因。

## 9. EMA 和 Adam 的关系

Adam 里有两个移动平均。

第一个是梯度的一阶矩，也就是梯度本身的指数加权移动平均：

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
$$

它记录的是：最近一段时间梯度大概往哪个方向。

第二个是梯度平方的指数加权移动平均：

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
$$

它记录的是：最近一段时间梯度大小大不大。

Adam 再用这两个量一起决定参数更新：

$$
\theta_t=\theta_{t-1}-\eta\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}
$$

所以，如果没有先理解 EMA，Adam 的 $m_t$、$v_t$ 就会显得像凭空冒出来的两个公式。

## 10. EMA 和 BatchNorm 的关系

BatchNorm 在训练时使用当前 batch 的均值和方差：

$$
\mu_B,\quad \sigma_B^2
$$

但是推理时不能依赖当前 batch，所以需要保存训练过程中的 running mean 和 running variance。

这些 running 统计量通常也是用移动平均更新的：

$$
\mu_{running}\leftarrow \beta\mu_{running}+(1-\beta)\mu_B
$$

$$
\sigma^2_{running}\leftarrow \beta\sigma^2_{running}+(1-\beta)\sigma_B^2
$$

所以 BatchNorm 里的 running mean / running variance，也可以用 EMA 的思想来理解。

## 11. EMA 还可以用于模型参数

有些训练方法还会对模型参数本身做 EMA。

假设当前模型参数是：

$$
\theta_t
$$

维护一份 EMA 参数：

$$
\theta^{EMA}_t=\beta\theta^{EMA}_{t-1}+(1-\beta)\theta_t
$$

这相当于保存一份更平滑的模型参数。

为什么可能有用？

训练过程中的参数会受 batch 噪声影响而抖动，EMA 参数相当于把多个历史模型做了平滑平均，推理时有时会更稳定。

但入门阶段先不必急着使用它，只要知道 EMA 不只用于梯度，也可以用于统计量和模型参数。

## 12. 本节总结

这一节的逻辑链是：

```text
训练中的梯度和统计量经常很抖
-> 普通平均不够关注最近趋势
-> 指数加权移动平均让近期数据权重大、远期数据权重小
-> beta 控制记忆长短
-> beta 越大越平滑，但反应越慢
-> 刚开始会偏小，所以需要偏差修正
-> Momentum、Adam、BatchNorm running 统计量都用到了这个思想
```

先记住两个公式：

指数加权移动平均：

$$
v_t=\beta v_{t-1}+(1-\beta)x_t
$$

偏差修正：

$$
\hat{v}_t=\frac{v_t}{1-\beta^t}
$$

理解 EMA 后，再看 Momentum 和 Adam，就不是在背公式，而是在看“如何平滑梯度、如何稳定参数更新”。